In [2]:
import pandas as pd

# Load your preprocessed datasets
ratings_df = pd.read_csv('data/ratings.csv')
anime_df = pd.read_csv('data/anime.csv')
users_df = pd.read_csv('data/users.csv')

In [3]:
print("📄 ratings_df columns:")
print(ratings_df.columns.tolist())

print("\n📄 anime_df columns:")
print(anime_df.columns.tolist())

print("\n📄 users_df columns:")
print(users_df.columns.tolist())


📄 ratings_df columns:
['username', 'anime_id', 'my_watched_episodes', 'my_start_date', 'my_finish_date', 'rating', 'my_status', 'my_rewatching', 'my_rewatching_ep', 'my_last_updated', 'watch_year', 'watch_month', 'user_bias', 'anime_bias', 'adjusted_rating', 'total_ratings', 'mean_rating', 'rating_std', 'total_rewatches', 'normalized_rating', 'user_rating_count', 'anime_rating_count', 'days_since_start']

📄 anime_df columns:
['anime_id', 'title', 'title_english', 'title_japanese', 'title_synonyms', 'image_url', 'type', 'source', 'episodes', 'status', 'airing', 'aired_string', 'aired', 'duration', 'rating', 'score', 'scored_by', 'rank', 'popularity', 'members', 'favorites', 'premiered', 'broadcast', 'related', 'producer', 'licensor', 'studio', '  ', 'opening_theme', 'ending_theme', 'duration_min', 'aired_from_year', 'Action', 'Adventure', 'Cars', 'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game', 'Harem', 'Hentai', 'Historical', 'Horror', 'Josei', 'Kids', 'Magic', 'Mar

User-Based Collaborative Filtering

In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics import ndcg_score

# Import the updated UserCF model
from models.user_cf import UserCF

# --- Data Preprocessing ---
ratings_df['anime_id'] = ratings_df['anime_id'].astype(int)
ratings_df['rating'] = ratings_df['rating'].astype(float)

ratings_df['username_code'] = ratings_df['username'].astype('category').cat.codes
ratings_df['username_code'] = np.random.permutation(ratings_df['username_code'].values)
ratings_df['anime_code'] = ratings_df['anime_id'].astype('category').cat.codes

# Optional: Create mappings for later use
username_map = dict(enumerate(ratings_df['username'].astype('category').cat.categories))
anime_map = dict(enumerate(ratings_df['anime_id'].astype('category').cat.categories))

# Select a sample user code for evaluation
sample_username_code = ratings_df['username_code'].dropna().unique()[0]
sample_username = ratings_df[ratings_df['username_code'] == sample_username_code]['username'].iloc[0]

# --- Train User-Based Collaborative Filtering Model ---
user_cf_model = UserCF(ratings_df)
user_cf_model.train()

# --- Evaluation Function ---
def evaluate_model(model, user_id, k=10):
    recommended_items = model.recommend(user_id, top_k=k)
    actual_items = ratings_df[ratings_df['username_code'] == user_id]['anime_code'].values
    binary_relevance = [1 if item in actual_items else 0 for item in recommended_items[:k]]
    
    precision_at_k = sum(binary_relevance) / k
    recall_at_k = sum(binary_relevance) / (len(actual_items) or 1)
    ndcg_at_k = ndcg_score([binary_relevance], [binary_relevance], k=k)
    return precision_at_k, recall_at_k, ndcg_at_k

# --- Evaluate Model ---
precision, recall, ndcg = evaluate_model(user_cf_model, sample_username_code)
print(f"\n📊 Evaluation for user '{sample_username}' (code: {sample_username_code}):")

# --- Watched Anime ---
watched_anime_df = ratings_df[ratings_df['username_code'] == sample_username_code][['anime_id', 'rating']]
watched_anime_titles = anime_df[anime_df['anime_id'].isin(watched_anime_df['anime_id'])][['anime_id', 'title']]
watched_anime = watched_anime_df.merge(watched_anime_titles, on='anime_id', how='left')
watched_anime_sorted = watched_anime[['title', 'rating']].sort_values(by='rating', ascending=False)

print(f"\n🎬 Anime already watched by '{sample_username}':")
print(watched_anime_sorted)

# --- Recommended Anime ---
recommendations = user_cf_model.recommend(sample_username_code, top_k=10)
anime_id_map = dict(zip(ratings_df['anime_code'], ratings_df['anime_id']))
anime_ids = [anime_id_map.get(code) for code in recommendations if code in anime_id_map]

recommended_titles_df = anime_df[anime_df['anime_id'].isin(anime_ids)][['anime_id', 'title']]
ordered_titles = pd.DataFrame({'anime_id': anime_ids}).merge(recommended_titles_df, on='anime_id', how='left')

print(f"\n✨ Top 10 Anime Recommended for '{sample_username}':")
print(ordered_titles['title'].tolist())


📊 Evaluation for user 'Niev-sama' (code: 18146):

🎬 Anime already watched by 'Niev-sama':
                                                title  rating
18  Fate/stay night: Unlimited Blade Works 2nd Season    10.0
11                   Fullmetal Alchemist: Brotherhood     9.0
13                          Toaru Kagaku no Railgun S     9.0
12                        Kuroko no Basket 3rd Season     9.0
8                        Papa no Iukoto wo Kikinasai!     8.0
1                                  Shingeki no Kyojin     8.0
10                                  Zankyou no Terror     8.0
9                                          Koi to Uso     8.0
0                                  Mawaru Penguindrum     8.0
7                                       Mononoke Hime     8.0
6                                           Genshiken     8.0
5                                   Sukitte Ii na yo.     8.0
4                                Haiyore! Nyaruko-san     8.0
3                                       S

Content-Based Collaborative Filtering


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import importlib
import models.content_cf
importlib.reload(models.content_cf)
from models.content_cf import ContentBasedFiltering


def evaluate_content_based_model(ratings_df, anime_df, sample_size=5):
    """
    Evaluate Content-Based Filtering using a random sample of users.
    Outputs MSE/RMSE and a summary of watched & recommended anime for each sampled user.
    """
    cbf_model = ContentBasedFiltering(anime_df)

    # Sample a single user
    sample_users = np.random.choice(ratings_df['username'].unique(), 1, replace=False)

    true_ratings = []
    predicted_scores = []

    for username in sample_users:
        user_data = ratings_df[ratings_df['username'] == username]
        watched_ids = user_data['anime_id'].tolist()
        watched_titles = anime_df[anime_df['anime_id'].isin(watched_ids)]['title'].tolist()

        print(f"\n👤 User: {username}")
        print("📺 Watched Anime:")
        for title in watched_titles[:10]:  # limit to 10 titles
            print(f"   - {title}")

        for anime_id in watched_ids:
            rating = user_data[user_data['anime_id'] == anime_id]['rating'].values[0]
            try:
                pred_score = cbf_model.get_similarity_score(username, anime_id, ratings_df)
            except:
                pred_score = anime_df['normalized_score'].mean()  # fallback

            true_ratings.append(rating)
            predicted_scores.append(pred_score)

        # Recommend top 10 unseen anime
        all_anime_ids = set(anime_df['anime_id'])
        unseen_ids = list(all_anime_ids - set(watched_ids))
        recommendations = cbf_model.recommend(username, ratings_df, unseen_ids, top_n=10)

        print("\n🎯 Recommended Anime:")
        for anime_id, score in recommendations:
            title = anime_df[anime_df['anime_id'] == anime_id]['title'].values[0]
            print(f"   - {title} (score: {score:.3f})")

    # Normalize ratings to 0-1 for fair MSE comparison
    true_ratings = np.array(true_ratings) / 10.0
    predicted_scores = np.array(predicted_scores)

    mse = mean_squared_error(true_ratings, predicted_scores)
    rmse = np.sqrt(mse)

    print(f"\n📊 Overall Evaluation (on {sample_size} users):")
    print(f"🔹 MSE:  {mse:.4f}")
    print(f"🔹 RMSE: {rmse:.4f}")

if __name__ == "__main__":
    ratings_df = pd.read_csv("data/ratings.csv")
    anime_df = pd.read_csv("data/anime.csv")

    evaluate_content_based_model(ratings_df, anime_df, sample_size=5)


👤 User: Lizzard998
📺 Watched Anime:
   - Shingeki no Kyojin
   - Bleach
   - Code Geass: Hangyaku no Lelouch R2
   - Boku no Hero Academia
   - Psycho-Pass
   - Hunter x Hunter (2011)
   - One Piece
   - Cowboy Bebop
   - Sen to Chihiro no Kamikakushi
   - Neon Genesis Evangelion

🎯 Recommended Anime:
   - Boku no Hero Academia 3rd Season (score: 0.600)
   - Boku no Hero Academia 2nd Season (score: 0.598)
   - Steins;Gate 0 (score: 0.597)
   - Made in Abyss (score: 0.597)
   - Gintama° (score: 0.590)
   - Kimi no Na wa. (score: 0.588)
   - Kizumonogatari III: Reiketsu-hen (score: 0.587)
   - Haikyuu!!: Karasuno Koukou VS Shiratorizawa Gakuen Koukou (score: 0.586)
   - Shingeki no Kyojin Season 2 (score: 0.585)
   - Kizumonogatari II: Nekketsu-hen (score: 0.585)

📊 Overall Evaluation (on 5 users):
🔹 MSE:  0.0695
🔹 RMSE: 0.2636


In [2]:
from sklearn.metrics import precision_score, recall_score
import numpy as np

def calculate_precision_at_k(recommended_items, watched_items, k=10):
    """Calculate Precision@K."""
    recommended_set = set([item for item, _ in recommended_items[:k]])
    watched_set = set(watched_items)
    intersection = recommended_set & watched_set
    return len(intersection) / k

def calculate_recall_at_k(recommended_items, watched_items, k=10):
    """Calculate Recall@K."""
    recommended_set = set([item for item, _ in recommended_items[:k]])
    watched_set = set(watched_items)
    intersection = recommended_set & watched_set
    return len(intersection) / len(watched_set)

def calculate_ndcg_at_k(recommended_items, watched_items, k=10):
    """Calculate NDCG@K."""
    recommended_ids = [item for item, _ in recommended_items[:k]]
    relevance_scores = [1 if item in watched_items else 0 for item in recommended_ids]
    
    # Calculate DCG
    dcg = 0
    for i, rel in enumerate(relevance_scores):
        dcg += (2 ** rel - 1) / np.log2(i + 2)  # Discounted cumulative gain
        
    # Calculate IDCG (Ideal DCG)
    ideal_relevance_scores = [1] * min(k, len(watched_items))  # Max relevance is 1 for all watched items
    idcg = 0
    for i, rel in enumerate(ideal_relevance_scores):
        idcg += (2 ** rel - 1) / np.log2(i + 2)
    
    # NDCG is the ratio of DCG to IDCG
    return dcg / idcg if idcg > 0 else 0

def evaluate_content_based_model(ratings_df, anime_df, sample_size=5, k=10):
    """
    Evaluate Content-Based Filtering using a random sample of users.
    Outputs MSE/RMSE, Precision@K, Recall@K, NDCG@K, and a summary of watched & recommended anime for each sampled user.
    """
    cbf_model = ContentBasedFiltering(anime_df)

    # Sample a single user
    sample_users = np.random.choice(ratings_df['username'].unique(), sample_size, replace=False)

    true_ratings = []
    predicted_scores = []
    
    precision_at_k = []
    recall_at_k = []
    ndcg_at_k = []

    for username in sample_users:
        user_data = ratings_df[ratings_df['username'] == username]
        watched_ids = user_data['anime_id'].tolist()
        watched_titles = anime_df[anime_df['anime_id'].isin(watched_ids)]['title'].tolist()

        print(f"\n👤 User: {username}")
        print("📺 Watched Anime:")
        for title in watched_titles[:10]:  # limit to 10 titles
            print(f"   - {title}")

        for anime_id in watched_ids:
            rating = user_data[user_data['anime_id'] == anime_id]['rating'].values[0]
            try:
                pred_score = cbf_model.get_similarity_score(username, anime_id, ratings_df)
            except:
                pred_score = anime_df['normalized_score'].mean()  # fallback

            true_ratings.append(rating)
            predicted_scores.append(pred_score)

        # Recommend top K unseen anime
        all_anime_ids = set(anime_df['anime_id'])
        unseen_ids = list(all_anime_ids - set(watched_ids))
        recommendations = cbf_model.recommend(username, ratings_df, unseen_ids, top_n=k)

        print("\n🎯 Recommended Anime:")
        for anime_id, score in recommendations:
            title = anime_df[anime_df['anime_id'] == anime_id]['title'].values[0]
            print(f"   - {title} (score: {score:.3f})")

        # Calculate Precision@K, Recall@K, NDCG@K
        precision_at_k.append(calculate_precision_at_k(recommendations, watched_ids, k))
        recall_at_k.append(calculate_recall_at_k(recommendations, watched_ids, k))
        ndcg_at_k.append(calculate_ndcg_at_k(recommendations, watched_ids, k))

    # Normalize ratings to 0-1 for fair MSE comparison
    true_ratings = np.array(true_ratings) / 10.0
    predicted_scores = np.array(predicted_scores)

    mse = mean_squared_error(true_ratings, predicted_scores)
    rmse = np.sqrt(mse)

    # Average metrics for all sampled users
    avg_precision_at_k = np.mean(precision_at_k)
    avg_recall_at_k = np.mean(recall_at_k)
    avg_ndcg_at_k = np.mean(ndcg_at_k)

    print(f"\n📊 Overall Evaluation (on {sample_size} users):")
    print(f"🔹 MSE:  {mse:.4f}")
    print(f"🔹 RMSE: {rmse:.4f}")
    print(f"🔹 Precision@{k}: {avg_precision_at_k:.4f}")
    print(f"🔹 Recall@{k}: {avg_recall_at_k:.4f}")
    print(f"🔹 NDCG@{k}: {avg_ndcg_at_k:.4f}")

if __name__ == "__main__":
    ratings_df = pd.read_csv("data/ratings.csv")
    anime_df = pd.read_csv("data/anime.csv")

    evaluate_content_based_model(ratings_df, anime_df, sample_size=5, k=10)



👤 User: A-Jay
📺 Watched Anime:
   - Sword Art Online
   - Toradora!
   - Tengen Toppa Gurren Lagann
   - Hunter x Hunter (2011)
   - Mahou Shoujo Madoka★Magica
   - Hataraku Maou-sama!
   - Kono Subarashii Sekai ni Shukufuku wo!
   - Gintama
   - Ore no Imouto ga Konnani Kawaii Wake ga Nai
   - FLCL

🎯 Recommended Anime:
   - Gintama° (score: 0.604)
   - Boku no Hero Academia 3rd Season (score: 0.602)
   - Boku no Hero Academia 2nd Season (score: 0.600)
   - Steins;Gate 0 (score: 0.596)
   - Kono Subarashii Sekai ni Shukufuku wo! 2 (score: 0.595)
   - Gintama&#039; (score: 0.591)
   - Boku no Hero Academia (score: 0.591)
   - Made in Abyss (score: 0.590)
   - Gintama&#039;: Enchousen (score: 0.590)
   - Violet Evergarden (score: 0.590)

👤 User: Yashido
📺 Watched Anime:
   - Steins;Gate
   - Code Geass: Hangyaku no Lelouch
   - Mirai Nikki (TV)
   - Sword Art Online II
   - Tengen Toppa Gurren Lagann
   - Kimi no Na wa.
   - Hunter x Hunter (2011)
   - Sen to Chihiro no Kamikakushi
   

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from models.content_cf import ContentBasedFiltering

def evaluate_content_based_model(ratings_df, anime_df, top_n=10):
    """
    Evaluate Content-Based Filtering using RMSE + Top-N Recommendation Accuracy
    """
    cbf_model = ContentBasedFiltering(anime_df)
    
    true_ratings = []
    predicted_scores = []

    all_precision = []
    all_recall = []

    users = ratings_df['username'].unique()

    for user in users:
        user_data = ratings_df[ratings_df['username'] == user]
        watched_anime_ids = set(user_data['anime_id'])
        liked_anime_ids = set(user_data[user_data['rating'] >= 7]['anime_id'])

        # Candidate anime: Not watched yet
        candidate_anime = anime_df[~anime_df['anime_id'].isin(watched_anime_ids)]

        recommendations = cbf_model.recommend(user, candidate_anime, ratings_df, top_n=top_n)
        recommended_anime_ids = set([anime_id for anime_id, _ in recommendations])

        # Evaluate Top-N Recommendation
        true_positives = len(recommended_anime_ids & liked_anime_ids)
        precision = true_positives / top_n if top_n else 0
        recall = true_positives / len(liked_anime_ids) if liked_anime_ids else 0

        all_precision.append(precision)
        all_recall.append(recall)

        # Optional: Also add RMSE-style scoring for similarity
        for anime_id in user_data['anime_id']:
            rating = user_data[user_data['anime_id'] == anime_id]['rating'].values[0]
            try:
                pred_score = cbf_model.get_similarity_score(user, anime_id, ratings_df)
            except:
                pred_score = anime_df['normalized_score'].mean()

            true_ratings.append(rating / 10.0)
            predicted_scores.append(pred_score)

    # Final evaluation metrics
    mse = mean_squared_error(true_ratings, predicted_scores)
    rmse = np.sqrt(mse)
    avg_precision = np.mean(all_precision)
    avg_recall = np.mean(all_recall)

    print(f"\n📊 Content-Based Filtering Evaluation")
    print(f"🔹 MSE:     {mse:.4f}")
    print(f"🔹 RMSE:    {rmse:.4f}")
    print(f"🔹 Precision@{top_n}: {avg_precision:.4f}")
    print(f"🔹 Recall@{top_n}:    {avg_recall:.4f}")


Neural Net Based Collaborative Filtering

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import FeatureUnion

# ----- Step 1: Dataset -----
class AnimeRatingDataset(Dataset):
    def __init__(self, user_ids, anime_ids, anime_contents, ratings):
        self.user_ids = user_ids
        self.anime_ids = anime_ids
        self.anime_contents = anime_contents
        self.ratings = ratings

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.user_ids[idx], dtype=torch.long),
            torch.tensor(self.anime_ids[idx], dtype=torch.long),
            torch.tensor(self.anime_contents[idx], dtype=torch.float32),
            torch.tensor(self.ratings[idx], dtype=torch.float32).unsqueeze(0)
        )

# ----- Step 2: Model -----
class HybridRecommender(nn.Module):
    def __init__(self, num_users, num_anime, content_dim):
        super(HybridRecommender, self).__init__()
        self.user_embedding = nn.Embedding(num_users, 100)  # Reduced embedding size
        self.anime_embedding = nn.Embedding(num_anime, 10)  # Reduced embedding size

        self.fc1 = nn.Linear(100 + 10 + content_dim, 128)  # Reduced hidden layer size
        self.fc2 = nn.Linear(128, 64)  # Reduced hidden layer size
        self.output = nn.Linear(64, 1)

        self.relu = nn.ReLU()
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, user_id, anime_id, anime_content):
        user_embed = self.user_embedding(user_id)
        anime_embed = self.anime_embedding(anime_id)
        x = torch.cat([user_embed, anime_embed, anime_content], dim=1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.output(x)

# ----- Step 3: Preprocessing Content Features -----
def prepare_content_features(anime_df):
    anime_df = anime_df.fillna('')

    # Genre columns (already one-hot encoded)
    genre_cols = ['Action', 'Adventure', 'Cars', 'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi',
                  'Fantasy', 'Game', 'Harem', 'Hentai', 'Historical', 'Horror', 'Josei', 'Kids',
                  'Magic', 'Martial Arts', 'Mecha', 'Military', 'Music', 'Mystery', 'Parody', 
                  'Police', 'Psychological', 'Romance', 'Samurai', 'School', 'Sci-Fi', 'Seinen',
                  'Shoujo', 'Shoujo Ai', 'Shounen', 'Shounen Ai', 'Slice of Life', 'Space', 'Sports',
                  'Super Power', 'Supernatural', 'Thriller', 'Vampire']

    genre_encoded = anime_df[genre_cols].values.astype(np.float32)

    # TF-IDF on combined text field
    tfidf = TfidfVectorizer(max_features=100)
    tfidf_matrix = tfidf.fit_transform(anime_df['combined_text'].fillna('')).toarray()

    # Combine genre + TF-IDF
    combined_features = np.hstack([genre_encoded, tfidf_matrix])
    return combined_features, tfidf

# ----- Step 4: Encoding Users and Anime -----
def encode_ids(ratings_df, anime_df):
    user_encoder = LabelEncoder()
    anime_encoder = LabelEncoder()

    user_ids = user_encoder.fit_transform(ratings_df['username'])
    anime_ids = anime_encoder.fit_transform(ratings_df['anime_id'])

    ratings = ratings_df['rating'].values.astype(np.float32) / 10.0  # normalize ratings

    anime_df['anime_id_enc'] = anime_encoder.transform(anime_df['anime_id'])
    return user_ids, anime_ids, ratings, anime_encoder, user_encoder

# ----- Step 5: Train Model -----
def train_model():
    ratings_df = pd.read_csv("data/ratings.csv")
    anime_df = pd.read_csv("data/anime.csv")

    # Preprocess content features
    anime_contents, tfidf_vectorizer = prepare_content_features(anime_df)

    # Encode users and anime
    user_ids, anime_ids, ratings, anime_encoder, user_encoder = encode_ids(ratings_df, anime_df)

    # Sample a subset of users to speed up training (e.g., 5000 users)
    sampled_users = np.random.choice(np.unique(user_ids), 5000, replace=False)
    sampled_ratings_df = ratings_df[ratings_df['username'].isin(sampled_users)]

    # Match content features to rating rows
    anime_id_map = dict(zip(anime_df['anime_id_enc'], anime_contents))
    content_matrix = np.array([anime_id_map[aid] for aid in anime_ids])

    # Dataset and DataLoader
    dataset = AnimeRatingDataset(user_ids, anime_ids, content_matrix, ratings)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    # Model, optimizer, loss
    num_users = len(user_encoder.classes_)
    num_anime = len(anime_encoder.classes_)
    content_dim = anime_contents.shape[1]

    model = HybridRecommender(num_users, num_anime, content_dim)
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    # Early stopping parameters
    best_loss = float('inf')
    patience = 3
    trigger_times = 0

    # Train loop
    for epoch in range(1, 4):  # Reduced epochs to 20
        model.train()
        total_loss = 0

        for user_id, anime_id, anime_content, rating in loader:
            optimizer.zero_grad()
            preds = model(user_id, anime_id, anime_content)
            loss = criterion(preds, rating)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch}/20 - Loss: {avg_loss:.4f}")

        # Early stopping
        if avg_loss < best_loss:
            best_loss = avg_loss
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("Early stopping triggered")
                break

    return model, anime_encoder, user_encoder, anime_contents

# ----- Entry Point -----
if __name__ == "__main__":
    model, anime_enc, user_enc, anime_contents = train_model()

def recommend_top_n(user_id, anime_df, ratings_df, model, user_encoder, anime_encoder, tfidf_vectorizer, n=10):
    """
    Recommend top N animes for a user using the trained hybrid model.
    """
    model.eval()  # Set the model to evaluation mode

    # Step 1: Get watched anime for the given user
    watched_anime_ids = ratings_df[ratings_df['username'] == user_id]['anime_id'].values

    # Step 2: Filter out watched animes to get candidate animes for recommendation
    candidate_anime = anime_df[~anime_df['anime_id'].isin(watched_anime_ids)]

    # Initialize lists for anime IDs, user IDs, and anime content features
    anime_ids = []
    user_ids = []
    anime_contents = []

    # Step 3: Prepare the content (TF-IDF features for the synopsis and genre features)
    genre_cols = [col for col in anime_df.columns if col not in ['anime_id', 'title', 'combined_text']]  # Genre columns
    for _, row in candidate_anime.iterrows():
        anime_ids.append(row['anime_id'])
        user_ids.append(user_id)

        # Get TF-IDF features from the combined text (synopsis)
        tfidf_vec = tfidf_vectorizer.transform([row['combined_text']]).toarray()

        # Get genre features (one-hot encoding)
        genre_matrix = row[genre_cols].values.astype(np.float32)

        # Combine the genre features and TF-IDF features
        anime_contents.append(np.hstack([genre_matrix, tfidf_vec[0]]))  # Concatenate both

    # Step 4: Encode user and anime IDs using LabelEncoder
    user_indices = torch.tensor([user_encoder.transform([u])[0] for u in user_ids], dtype=torch.long)
    anime_indices = torch.tensor([anime_encoder.transform([a])[0] for a in anime_ids], dtype=torch.long)
    
    # Convert anime content to tensor
    content_tensor = torch.tensor(np.array(anime_contents), dtype=torch.float32)

    # Step 5: Get predictions from the model
    with torch.no_grad():
        preds = model(user_indices, anime_indices, content_tensor).squeeze().numpy()

    # Step 6: Sort predictions and get top N anime IDs
    top_n_indices = preds.argsort()[-n:][::-1]  # Get the indices of top N predicted ratings
    top_n_anime_ids = [anime_ids[i] for i in top_n_indices]

    # Step 7: Return top N recommended anime details
    return anime_df[anime_df['anime_id'].isin(top_n_anime_ids)][['anime_id', 'title', 'genre']]


Epoch 1/20 - Loss: 0.0143
Epoch 2/20 - Loss: 0.0117
Epoch 3/20 - Loss: 0.0114


Neural Net Based Collaborative Filtering




 We'll train and evaluate several recommendation models:
 - User-Based Collaborative Filtering
 - Item-Based Collaborative Filtering  
 - SVD Matrix Factorization
 - Neural Collaborative Filtering
 - Content-Based Filtering
 - LightFM Hybrid Model
 - Implicit ALS
